### In this notebook I tried a different approach with combining the datasets and did some EDA with the whole set

In [2]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import average_precision_score

In [3]:
# Combining the datasets together with a new column for run_id and road_id

dfA1 = pd.read_csv("../dataset/cleaned/Iisingintie0.csv")
dfA2 = pd.read_csv("../dataset/cleaned/Iisingintie1.csv")
dfA3 = pd.read_csv("../dataset/cleaned/Iisingintie2.csv")

dfB1 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie0.csv")
dfB2 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie1.csv")
dfB3 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie2.csv")
dfB4 = pd.read_csv("../dataset/cleaned/Sinetänsalmentie3.csv")

dfC1 = pd.read_csv("../dataset/cleaned/Kaittiaisentie0.csv")


runs = {
    "A1": (dfA1, "A"), "A2": (dfA2, "A"), "A3": (dfA3, "A"),
    "B1": (dfB1, "B"), "B2": (dfB2, "B"), "B3": (dfB3, "B"), "B4": (dfB4, "B"),
    # "C1": (dfC1, "C"),
}

df = pd.concat(
    [d.assign(run_id=rid, road_id=road) for rid, (d, road) in runs.items()],
    ignore_index=True,
)

df["AccZ_max"] = df["AccZ"].max()

In [4]:
df

,Timer(ms),Loc(m),CElev(m),Roll(deg),Pitch(deg),Yaw(deg),AccX,AccY,AccZ,GyrX,GyrY,GyrZ,Speed_kmh,Frostbumps,run_id,road_id,AccZ_max
0,45493158.5,3850.503,106.794000,1.1747,-3.0780,141.0204,1.53481,0.05712,9.58495,0.03196,0.00072,-0.00300,62.1936,0,A1,A,23.70313
1,45493161.0,3850.546,106.794000,1.2245,-3.0428,141.1350,1.51066,-0.04196,10.50449,0.02201,0.00598,-0.01390,62.2044,0,A1,A,23.70313
2,45493163.5,3850.589,106.794000,1.2281,-3.0428,141.1349,1.15831,0.37071,10.76888,0.01993,0.00065,-0.00562,62.2116,0,A1,A,23.70313
3,45493166.0,3850.633,106.794000,1.2330,-3.0430,141.1341,1.03301,0.31102,11.42271,0.02887,-0.00077,-0.01012,62.2224,0,A1,A,23.70313
4,45493168.5,3850.676,106.794000,1.2363,-3.0431,141.1329,0.99574,-0.37334,11.76243,0.01783,0.00007,-0.01306,62.2332,0,A1,A,23.70313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288458,41563778.5,5512.852,10.355333,0.6512,-1.5795,-48.0806,-1.78920,0.16438,10.33153,-0.00936,-0.01235,-0.02487,33.9372,0,B4,B,23.70313
288459,41563781.0,5512.875,11.012904,0.6481,-1.5840,-47.9881,-1.88369,-0.02342,10.76440,-0.00575,-0.00767,-0.02849,33.9372,0,B4,B,23.70313
288460,41563783.5,5512.899,10.800525,0.6452,-1.5904,-47.8951,-1.92509,-0.06587,10.62556,-0.00451,-0.02198,-0.02490,33.9372,0,B4,B,23.70313
288461,41563786.0,5512.922,10.374600,0.6427,-1.5977,-47.8027,-1.60182,0.24200,10.34438,-0.00214,-0.02739,-0.02887,33.9408,0,B4,B,23.70313


In [5]:
run_names = list(runs.keys())
print(run_names)

['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4']


In [6]:
df["AccMagnitude"] = np.sqrt(
    df["AccX"]**2 +
    df["AccY"]**2 +
    df["AccZ"]**2
)

df["GyrMagnitude"] = np.sqrt(
    df["GyrX"]**2 +
    df["GyrY"]**2 +
    df["GyrZ"]**2
)

In [7]:
feature_cols = ["AccX", "AccY", "AccZ", "GyrX", "GyrY", "GyrZ",
                "Pitch(deg)", "Roll(deg)", "Speed_kmh", "RBCSV_80(cf)", "AccMagnitude", "GyrMagnitude", "AccZ_max"]


In [8]:
def make_row_windows(run_df, win=0, stride=0, min_percentage=0.1):
    run_df = run_df.sort_values("Timer(ms)")

    X = run_df[feature_cols].to_numpy(np.float32)
    y = run_df["Frostbumps"].to_numpy()

    X_clips, y_clips = [], []

    for start in range(0, len(X) - win + 1, stride):
        end = start + win

        X_clips.append(X[start:end])

        bump_percentage = y[start:end].mean()
        y_clips.append(int(bump_percentage >= min_percentage))

    return np.array(X_clips), np.array(y_clips)

In [9]:
def get_clips(run_names):
    all_X, all_y = [], []
    for name in run_names:
        one_run = df[df["run_id"] == name]
        X, y = make_row_windows(one_run, win=200, stride=150)
        all_X.append(X)
        all_y.append(y)
    return np.concatenate(all_X), np.concatenate(all_y)

In [10]:
# Checking the number of windows per run

for run in ["A1", "A2", "A3", "B1", "B2", "B3", "B4"]:
    X, y = get_clips([run])

    print(
        f"{run}: {len(y)} windows, "
        f"{y.mean():.1%} positive"
    )

KeyError: "['RBCSV_80(cf)'] not in index"

In [ ]:
# Training, validation, test splits

Xtr, ytr = get_clips(["A3","B3","B4","B1"])
Xva, yva = get_clips(["A2","B2"])
Xte, yte = get_clips(["A1"])

print(Xtr.shape, Xva.shape, Xte.shape)
print(ytr.mean(), yva.mean(), yte.mean())

(987, 200, 13) (552, 200, 13) (378, 200, 13)
0.25430597771023306 0.24456521739130435 0.23809523809523808


In [ ]:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler

# n_features = Xtr.shape[-1]      # 9 now

# scaler = StandardScaler()
# scaler.fit(Xtr.reshape(-1, n_features))

# Xtr = scaler.transform(Xtr.reshape(-1, n_features)).reshape(Xtr.shape)
# Xva = scaler.transform(Xva.reshape(-1, n_features)).reshape(Xva.shape)
# Xte = scaler.transform(Xte.reshape(-1, n_features)).reshape(Xte.shape)

## XGBoost stuff starts here


In [ ]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import ParameterSampler
# from sklearn.metrics import average_precision_score
# from scipy.stats import randint, uniform


# # Flatten windows
# Xtr_xgb = Xtr.reshape(Xtr.shape[0], -1)
# Xva_xgb = Xva.reshape(Xva.shape[0], -1)
# Xte_xgb = Xte.reshape(Xte.shape[0], -1)

# param_grid = {
#     "n_estimators": randint(200, 1001),
#     "max_depth": randint(2, 8),
#     "learning_rate": uniform(0.01, 0.19),
#     "min_child_weight": randint(1, 11),
#     "subsample": uniform(0.6, 0.4),
#     "colsample_bytree": uniform(0.6, 0.4),
#     "gamma": uniform(0, 2),
#     "reg_lambda": uniform(0.5, 9.5),
#     "reg_alpha": uniform(0, 2)
# }

# candidates = list(
#     ParameterSampler(
#         param_grid,
#         n_iter=35,
#         random_state=69
#     )
# )

# results = []

# for i, params in enumerate(candidates):

#     model = XGBClassifier(
#         **params,
#         eval_metric="aucpr",
#         random_state=69,
#         n_jobs=-1
#     )

#     model.fit(Xtr_xgb, ytr)

#     val_prob = model.predict_proba(Xva_xgb)[:, 1]

#     pr_auc = average_precision_score(yva, val_prob)

#     results.append({
#         **params,
#         "PR_AUC": pr_auc
#     })

#     print(f"Model {i+1}/35 | Validation PR-AUC: {pr_auc:.4f}")



#     results_df = pd.DataFrame(results)

# results_df = results_df.sort_values(
#     by="PR_AUC",
#     ascending=False
# )

# print(results_df.head(10))

In [ ]:
# best_params = results_df.iloc[0].drop("PR_AUC").to_dict()

# best_params["n_estimators"] = int(best_params["n_estimators"])
# best_params["max_depth"] = int(best_params["max_depth"])
# best_params["min_child_weight"] = int(best_params["min_child_weight"])

# best_xgb = XGBClassifier(
#     **best_params,
#     eval_metric="aucpr",
#     random_state=69,
#     n_jobs=-1
# )

# best_xgb.fit(Xtr_xgb, ytr)

# print("Best parameters:")
# print(best_params)

In [ ]:
# val_prob = best_xgb.predict_proba(Xva_xgb)[:, 1]

# print("Validation PR-AUC:",
#       average_precision_score(yva, val_prob))

# print("Validation ROC-AUC:",
#       roc_auc_score(yva, val_prob))

# # Test
# test_prob = best_xgb.predict_proba(Xte_xgb)[:, 1]
# test_pred = (test_prob >= 0.5).astype(int)

# print("\nTest PR-AUC:",
#       average_precision_score(yte, test_prob))

# print("Test ROC-AUC:",
#       roc_auc_score(yte, test_prob))

# print("\nConfusion matrix:")
# print(confusion_matrix(yte, test_pred))

# print("\nClassification report:")
# print(classification_report(yte, test_pred))

In [ ]:
from xgboost import XGBClassifier

# Flatten windows
Xtr_xgb = Xtr.reshape(Xtr.shape[0], -1)
Xva_xgb = Xva.reshape(Xva.shape[0], -1)
Xte_xgb = Xte.reshape(Xte.shape[0], -1)



# Create model
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    eval_metric="aucpr",
    random_state=69,
    n_jobs=-1
)

# Train
xgb.fit(Xtr_xgb, ytr)

# Validation
val_prob = xgb.predict_proba(Xva_xgb)[:, 1]


NameError: name 'Xtr' is not defined

In [ ]:
print("Validation PR-AUC:",
      average_precision_score(yva, val_prob))

print("Validation ROC-AUC:",
      roc_auc_score(yva, val_prob))

# Test
test_prob = xgb.predict_proba(Xte_xgb)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("\nTest PR-AUC:",
      average_precision_score(yte, test_prob))

print("Test ROC-AUC:",
      roc_auc_score(yte, test_prob))

print("Balanced Accuracy Score",
      balanced_accuracy_score(yte, test_pred))

print("\nConfusion matrix:")
print(confusion_matrix(yte, test_pred))

print("\nClassification report:")
print(classification_report(yte, test_pred))




Validation PR-AUC: 0.7328224630438386
Validation ROC-AUC: 0.8622613020694555

Test PR-AUC: 0.7664656147286111
Test ROC-AUC: 0.8925154320987655

Confusion matrix:
[[207  81]
 [ 12  78]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      0.72      0.82       288
           1       0.49      0.87      0.63        90

    accuracy                           0.75       378
   macro avg       0.72      0.79      0.72       378
weighted avg       0.84      0.75      0.77       378



In [ ]:
df.columns

Index(['Timer(ms)', 'Loc(m)', 'CElev(m)', 'Roll(deg)', 'Pitch(deg)',
       'Yaw(deg)', 'AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ',
       'RBCSV_80(cf)', 'Speed_kmh', 'Frostbumps', 'run_id', 'road_id',
       'AccZ_max', 'AccMagnitude', 'GyrMagnitude'],
      dtype='str')

In [ ]:
df

,Timer(ms),Loc(m),CElev(m),Roll(deg),Pitch(deg),Yaw(deg),AccX,AccY,AccZ,GyrX,GyrY,GyrZ,RBCSV_80(cf),Speed_kmh,Frostbumps,run_id,road_id,AccZ_max,AccMagnitude,GyrMagnitude
0,45493158.5,3850.503,106.794000,1.1747,-3.0780,141.0204,1.53481,0.05712,9.58495,0.03196,0.00072,-0.00300,0.16845,62.1936,0,A1,A,23.70313,9.707223,0.032109
1,45493161.0,3850.546,106.794000,1.2245,-3.0428,141.1350,1.51066,-0.04196,10.50449,0.02201,0.00598,-0.01390,0.16846,62.2044,0,A1,A,23.70313,10.612642,0.026710
2,45493163.5,3850.589,106.794000,1.2281,-3.0428,141.1349,1.15831,0.37071,10.76888,0.01993,0.00065,-0.00562,0.16846,62.2116,0,A1,A,23.70313,10.837338,0.020717
3,45493166.0,3850.633,106.794000,1.2330,-3.0430,141.1341,1.03301,0.31102,11.42271,0.02887,-0.00077,-0.01012,0.16845,62.2224,0,A1,A,23.70313,11.473541,0.030602
4,45493168.5,3850.676,106.794000,1.2363,-3.0431,141.1329,0.99574,-0.37334,11.76243,0.01783,0.00007,-0.01306,0.16843,62.2332,0,A1,A,23.70313,11.810404,0.022102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288458,41563778.5,5512.852,10.355333,0.6512,-1.5795,-48.0806,-1.78920,0.16438,10.33153,-0.00936,-0.01235,-0.02487,1.32488,33.9372,0,B4,B,23.70313,10.486600,0.029303
288459,41563781.0,5512.875,11.012904,0.6481,-1.5840,-47.9881,-1.88369,-0.02342,10.76440,-0.00575,-0.00767,-0.02849,1.32491,33.9372,0,B4,B,23.70313,10.927998,0.030059
288460,41563783.5,5512.899,10.800525,0.6452,-1.5904,-47.8951,-1.92509,-0.06587,10.62556,-0.00451,-0.02198,-0.02490,1.32494,33.9372,0,B4,B,23.70313,10.798742,0.033518
288461,41563786.0,5512.922,10.374600,0.6427,-1.5977,-47.8027,-1.60182,0.24200,10.34438,-0.00214,-0.02739,-0.02887,1.32495,33.9408,0,B4,B,23.70313,10.470463,0.039853


In [ ]:
from sklearn.metrics import confusion_matrix

test_prob = xgb.predict_proba(Xte_xgb)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

false_positives = np.where((yte == 0) & (test_pred == 1))[0]
false_negatives = np.where((yte == 1) & (test_pred == 0))[0]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 81
False negatives: 12


In [ ]:
import pandas as pd

n_features = len(feature_cols)

importance = pd.DataFrame({
    "Feature": feature_cols * Xtr.shape[1],
    "Importance": xgb.feature_importances_
})

print(
    importance.groupby("Feature")["Importance"]
    .sum()
    .sort_values(ascending=False)
)

Feature
RBCSV_80(cf)    0.198260
GyrMagnitude    0.155096
AccX            0.115656
AccY            0.083039
GyrY            0.067254
AccMagnitude    0.062947
GyrZ            0.062615
Pitch(deg)      0.052589
AccZ            0.052265
GyrX            0.051283
Speed_kmh       0.049508
Roll(deg)       0.049487
AccZ_max        0.000000
Name: Importance, dtype: float32
